# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring** (Lane 2).

Question: Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

I'm choosing this lane because it produces a ranked, explainable queue — something someone can act on this week — rather than just a correlation finding (Lane 1) or a typology with no built-in urgency (Lane 3).
I also want hands-on practice with the full method stack this lane calls for: a transparent baseline score first, then checking whether logistic regression, a decision tree, random forest, or gradient boosting actually earns its extra complexity over that baseline; a ranked queue with reason codes attached to each item; and honest evaluation with precision@K, recall, average precision, plus a by-hand review of my top 20.
Keeping Lane 1 as my fallback if this framing doesn't hold up under the Week 3 leakage check.



In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Among a client's content items, which ones should a content editor
review for refresh *this week*, and in what order?

**Unit of analysis:** one content item (a page), scored independently within its client.

**Decision it improves:** which page(s) an editor spends their limited refresh hours on first,
out of potentially thousands of pages per client.

**Who acts, and what do they do:** a content editor or SEO lead at a FlyRank client. Given a
ranked queue, they open the top items and decide whether to rewrite, expand, prune, or leave
alone — a concrete action, not just a number.

**Output:** a ranked queue (content_id, score, reason code, suggested action).

**Cost of a wrong call:**
- *False positive* (we flag a page as urgent, but it wasn't really declining or worth fixing):
  wasted editor hours — the real cost is opportunity cost, since editor time is the scarcest
  resource in this workflow.
- *False negative* (a page is quietly losing traffic and we never surface it): a slower, silent
  loss of impressions/clicks that compounds the longer it goes unnoticed — harder to see, but
  more expensive over time.
Because the downside of a false positive is "wasted an hour," not "broke something," I don't need
a very conservative threshold — a queue that surfaces some borderline cases is fine, as long as
the ordering (most-urgent-first) is trustworthy.

**Why data/ML helps at all:** with 30,000 rows and multiple correlated signals (position, trend,
freshness, impressions, content type) that interact — e.g. "declining" only matters if there's
real demand behind it — a single hand-written rule ("if declining, refresh") is too blunt. A
transparent baseline score built from several signals at once, later checked against a model,
can rank pages far better than eyeballing one column at a time. This is still closer to scoring
and ranking than "machine learning" in the deep-model sense, and I say so explicitly below.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

import pandas as pd
import os

# Works locally (running from work/notebooks/) and in Colab (repo not cloned yet)
local_path = "../../data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    # Colab: clone the repo once, then read from it
    if not os.path.exists("week1-asm1"):
        !git clone https://github.com/UnzilaAhsan/week1-asm1.git
    df = pd.read_csv("week1-asm1/data/raw/content_refresh_anonymized.csv")

n = len(df)
print(f"Rows: {n:,}  |  Clients: {df['client_id'].nunique()}")


# 1) How much of the inventory is trending down at all?
down = df[df["trend_direction"] == "down"]
print(f"1) Rows trending down: {len(down):,} ({len(down)/n*100:.1f}% of all rows)")

# 2) Of those, how many combine a decline with real, measurable demand
#    (not just noise on a page nobody sees) and a position worth defending (top 20)?
#    avg_position == 0 means "no data", so it's excluded, not treated as rank 0.
decay_candidates = df[
    (df["trend_direction"] == "down")
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["impressions_90d"] > 50)
]
print(f"2) Declining + ranked top-20 + impressions_90d>50: {len(decay_candidates):,} "
      f"({len(decay_candidates)/n*100:.1f}%)")

# 3) A tighter, higher-priority slice: pages in striking distance (positions ~11-20,
#    i.e. page 2) that are trending down. These are the pages closest to a page-1
#    breakthrough that are instead sliding backward -- a natural "fix first" candidate.
striking_down = df[(df["position_tier"] == "striking") & (df["trend_direction"] == "down")]
print(f"3) 'Striking distance' pages trending down: {len(striking_down):,} "
      f"({len(striking_down)/n*100:.1f}%)")

# 4) Staleness context: how much of the inventory hasn't been touched in a long time
#    AND is still getting impressions (i.e. worth touching)?
stale_visible = df[df["freshness_tier"].isin(["91-180", "181+"]) & (df["impressions_last_30d"] > 0)]
print(f"4) Stale (90+ days) but still getting impressions: {len(stale_visible):,} "
      f"({len(stale_visible)/n*100:.1f}%)")


Rows: 30,000  |  Clients: 32
1) Rows trending down: 16,262 (54.2% of all rows)
2) Declining + ranked top-20 + impressions_90d>50: 9,967 (33.2%)
3) 'Striking distance' pages trending down: 4,452 (14.8%)
4) Stale (90+ days) but still getting impressions: 8,741 (29.1%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*


**What I can claim:**
- *Observed:* "54.2% of rows in the starter sample are tagged `down` in `trend_direction`" — a
  fact about this dataset, not a universal truth about search behavior.
- *Directional:* "Pages that are both declining and still receiving meaningful impressions
  (`impressions_90d > 50`) make up about a third of the sample — a plausible pool for a refresh
  queue" — a reasonable direction to build toward, not a proven causal effect.
- *Decision-support:* the eventual output is a ranked queue meant to help a human prioritize,
  not an automated decision — the editor still chooses what to do with each page.

**What I will never claim:**
- That refreshing a page *causes* traffic recovery — this is observational data; I never see the
  counterfactual (what would have happened without a refresh), so I can't make causal claims
  without a proper before/after or holdout test.
- That I am predicting a Google ranking factor or "how Google's algorithm works." I only have
  FlyRank's own measured search/analytics signals, not Google's internals.
- That `trend_direction` or `trend_pct` are discovered "signals" — they're precomputed from the
  same logic that defines the decline label, so treating them as a model feature/finding would be
  circular. I will treat them as label material only, not as evidence of a discovered pattern.
- That a high score guarantees a wrong call is costly — as discussed above, the cost of a false
  positive here is mild (wasted review time), which is part of *why* a scored, ranked approach is
  reasonable even before rigorous validation.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.